# 05b · European run

Champions League, Europa League and Conference League fixtures **between two clubs from the five leagues**, priced from the league model.

League → UEFA is one-way: this notebook reads the league model and writes only to `Outputs/Euro/`. Run `05_Run` first on the same day so the clean table is current; nothing here refreshes or changes it.

How a fixture is priced (`fpp.europe`): each side is predicted by the production league model as though it were a league game, with the opponent converted at par into its league, then the European layer applies the league exchange rates (goals, shots, SOT; corners stay at par). Settings are frozen in `fpp.europe.config.PRODUCTION`.

In [ ]:
import fpp
import pandas as pd
from fpp.europe import predict as ep, run as er

pd.set_option("display.max_columns", None)

In [ ]:
DATE_FROM = "13-10-2026"     # dd-mm-yyyy
DATE_TO   = "15-10-2026"

In [ ]:
tm  = fpp.clean.load_clean_table()
ctx = fpp.RunContext.load()
models = fpp.predict.train_production(tm, ctx)

In [ ]:
fx = ep.fixtures(pd.to_datetime(DATE_FROM, format="%d-%m-%Y"), pd.to_datetime(DATE_TO, format="%d-%m-%Y"))
fx = ep.eligible(fx, tm)
display(fx[["date", "comp", "stage", "home_team", "home_league", "away_team", "away_league"]])

In [ ]:
layers = ep.production_layers()
disp   = ep.dispersion(layers)
preds  = ep.score(fx, tm, models, layers, ctx)

cols = ["date", "league", "home_team", "away_team",
        "goals_home", "goals_away", "shots_home", "shots_away", "corners_home", "corners_away"]
display(preds[cols])

In [ ]:
props = fpp.staking.propositions(preds, disp)
qual  = fpp.staking.qualifying(props)

# Coloured against each competition's own realised rates, not a domestic league's.
paths = er.write_outputs(preds, qual, ep.competition_table(), disp)
for k, p in paths.items():
    print(f"{k:17} -> {p}")